<a href="https://colab.research.google.com/github/Kanabu1/Explainable-Fake-News-Detection/blob/main/app_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers
!pip install torch

In [2]:
!pip install -q streamlit
!npm install -g localtunnel


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 31.4 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏
added 22 packages in 6s
⠏
⠏3 packages are looking for funding
⠏  run `npm fund` for details
⠏

In [3]:
%%writefile app.py
import streamlit as st
import torch
import torch.nn.functional as F
from transformers import BertTokenizer, BertForSequenceClassification, RobertaTokenizer, RobertaForSequenceClassification

# --- Configuration ---
# Path to your best performing BERT model
ROBERTA_MODEL_PATH = "/content/drive/MyDrive/roberta_fake_news_detector2"




# Device Config
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- Predefined Examples from LIAR Dataset ---
EXAMPLES = {
    "Custom Input": {
        "statement": "",
        "speaker": "",
        "party": "",
        "context": ""
    },
    "Example 1 (Fake): Economy": {
        "statement": "The economy is crashing and unemployment has doubled since the last administration.",
        "speaker": "Blog Posting",
        "party": "None",
        "context": "Social Media"
    },
    "Example 2 (Real): Tax Policy": {
        "statement": "We have cut taxes for small businesses 18 times in the last five years.",
        "speaker": "Barack Obama",
        "party": "Democrat",
        "context": "Campaign Speech"
    },
    "Example 3 (Fake): Healthcare": {
        "statement": "The Affordable Care Act includes a provision that will mandate end-of-life counseling for seniors.",
        "speaker": "Sarah Palin",
        "party": "Republican",
        "context": "Facebook Post"
    },
    "Example 4 (Real): Education": {
        "statement": "The Chicago Bears have had more starting quarterbacks in the last 10 years than the total number of tenured faculty fired.",
        "speaker": "Robin Vos",
        "party": "Republican",
        "context": "Online Opinion Piece"
    }
}

# --- 1. Model Loading ---
@st.cache_resource
def load_model():
    print("Loading BERT model...")

    tokenizer = RobertaTokenizer.from_pretrained(ROBERTA_MODEL_PATH)
    model = RobertaForSequenceClassification.from_pretrained(ROBERTA_MODEL_PATH)



    model.to(device)
    model.eval()
    return tokenizer, model


# --- 2. Prediction Logic ---
def predict(statement, speaker, context, tokenizer, model):
    # Construct input string (Context Injection)
    spk = speaker if speaker else "Unknown"
    ctx = context if context else "General"
    classifier_input = f"{spk} stated in {ctx}: {statement}"

    # Tokenize
    inputs = tokenizer(
        classifier_input,
        return_tensors="pt",
        max_length=256,
        truncation=True,
        padding=True
    ).to(device)

    # Inference
    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=1)

        # 0 = Real, 1 = Fake (Based on your high purity mapping)
        pred_idx = torch.argmax(probs, dim=1).item()
        confidence = probs[0][pred_idx].item()

    labels = ["Real", "Fake"]
    return labels[pred_idx], confidence

# --- 3. UI Logic ---
st.set_page_config(page_title="Fake News Classifier", page_icon="🕵️")

st.title("🕵️ BERT Fake News Classifier")
st.markdown("This tool uses a fine-tuned **BERT** model to detect deceptive political statements based on text and context.")

# Initialize Session State for inputs if they don't exist
if 'statement' not in st.session_state: st.session_state['statement'] = ""
if 'speaker' not in st.session_state: st.session_state['speaker'] = ""
if 'context' not in st.session_state: st.session_state['context'] = ""
if 'party' not in st.session_state: st.session_state['party'] = ""

# Callback to update fields when dropdown changes
def update_fields():
    selection = st.session_state.example_select
    data = EXAMPLES[selection]
    st.session_state['statement'] = data['statement']
    st.session_state['speaker'] = data['speaker']
    st.session_state['context'] = data['context']
    st.session_state['party'] = data['party']

# --- Input Section ---
st.subheader("Test Data")

# Dropdown for Examples
st.selectbox(
    "Choose an Example or 'Custom Input':",
    options=list(EXAMPLES.keys()),
    key="example_select",
    on_change=update_fields
)

col1, col2 = st.columns(2)
with col1:
    speaker = st.text_input("Speaker", key="speaker")
    party = st.selectbox("Party", ["", "Democrat", "Republican", "Independent", "None"], key="party")
with col2:
    context = st.text_input("Context (e.g., Speech, Tweet)", key="context")

statement = st.text_area("Statement", height=100, key="statement")

# --- Prediction Section ---
if st.button("Analyze Statement", type="primary"):
    if not statement:
        st.warning("Please enter a statement.")
    else:
        tokenizer, model = load_model()

        if model:
            with st.spinner("Analyzing..."):
                # Combine speaker/party for the prompt
                full_speaker = f"{speaker} ({party})" if party else speaker

                label, conf = predict(statement, full_speaker, context, tokenizer, model)

                st.divider()

                # Dynamic Color for Result
                if label == "Fake":
                    st.error(f"### Result: **FAKE NEWS**")
                    st.progress(conf)
                    st.caption(f"Confidence: {conf:.1%}")
                else:
                    st.success(f"### Result: **REAL NEWS**")
                    st.progress(conf)
                    st.caption(f"Confidence: {conf:.1%}")

st.markdown("---")
st.caption("Trained on LIAR  Dataset")

Writing app.py


In [4]:
import urllib.request
print("Password/Endpoint IP for localtunnel is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

Password/Endpoint IP for localtunnel is: 34.80.94.33


In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦

⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴your url is: https://kind-experts-press.loca.lt

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.80.94.33:8501

2025-11-30 20:11:25.662471: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764533485.713661    9190 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764533485.729082    9190 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1764533485.770296    9190 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000

In [ ]:
import urllib.request
print("Password/Endpoint IP for localtunnel is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())
